# Choose a maximum plausible rate

Shows how an establishment's rate (here the turnover rate) is distributed so you can choose the highest
value treated as plausible. It looks at establishment-months with a rate that pass the active flag, with
one value per establishment and month so large workplaces do not dominate. The focus is on rates close
to 1 (almost everyone left): a table with bars of how many establishment-months sit in each narrow band,
then how many rows each candidate maximum would keep.

**Before running:** the flagged worker data exists in S3 and the parameters below are set. Data is read
lazily and only small aggregates are collected. This notebook only reads from S3; it writes nothing back.

**Stop the notebook instance when you finish** — SageMaker bills while it is running.

## Parameters

- `DATA_PATH` — the flagged worker data (S3 folder of parquet files).
- `STORAGE_OPTIONS` — Polars S3 options; leave `None` unless the region isn't found.
- `ESTABLISHMENT_COLUMN`, `MONTH_COLUMN` — together identify one establishment-month.
- `RATE_COLUMN` — the rate whose maximum is being chosen. Null rates are ignored.
- `ACTIVE_COLUMN` — boolean flag; only rows where it is true are used, as only those can be valid.
- `HISTOGRAM_FROM`, `HISTOGRAM_BIN_WIDTH` — the distribution table starts at this rate and uses bands this
  wide; a rate of exactly 1 gets its own row.
- `CANDIDATE_MAX_RATES` — maximum rates to compare. Example values; edit them.

In [ ]:
DATA_PATH = "s3://<bucket>/domain=<domain>/dataset=<dataset>/version=<version>/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
HISTOGRAM_FROM = 0.8
HISTOGRAM_BIN_WIDTH = 0.01
CANDIDATE_MAX_RATES = [0.8, 0.85, 0.9, 0.95, 0.98, 1.0]

# Column names are defined once, here. Edit them for your dataset.
ESTABLISHMENT_COLUMN = "establishment_id"
MONTH_COLUMN = "ascwds_worker_import_date"
RATE_COLUMN = "turnover_rate"
ACTIVE_COLUMN = "establishment_active"

## Setup

In [ ]:
import polars as pl

pl.Config.set_tbl_rows(40)

## One rate per establishment-month

Scans the data lazily, keeps the used columns, drops rows with a null rate or without the active flag, and
reduces to one row per establishment and month. The table counts the establishment-months used and shows
the rate at selected quantiles. Zero (nobody left) and one (everybody left) are shown separately because
they are the cases the plausibility flag is meant to catch.

In [ ]:
rate = pl.col(RATE_COLUMN)

rate_lf = (
    pl.scan_parquet(DATA_PATH, storage_options=STORAGE_OPTIONS)
    .select(ESTABLISHMENT_COLUMN, MONTH_COLUMN, RATE_COLUMN, ACTIVE_COLUMN)
    .filter(rate.is_not_null(), pl.col(ACTIVE_COLUMN))
    .select(ESTABLISHMENT_COLUMN, MONTH_COLUMN, RATE_COLUMN)
    .unique()
)

rate_lf.select(
    pl.len().alias("establishment_months"),
    (rate == 0).sum().alias("rate_exactly_0"),
    (rate == 1).sum().alias("rate_exactly_1"),
    *[rate.quantile(q).alias(f"p{round(q * 100)}") for q in (0.5, 0.9, 0.95, 0.99)],
).collect()

## Distribution near 1

Counts establishment-months in bands `HISTOGRAM_BIN_WIDTH` wide from `HISTOGRAM_FROM` up to 1, with
`1.0` (everybody left) as its own row. `share` is of all establishment-months used, so it shows how rare
each band is; `bar` draws the counts as blocks, scaled so the largest band is 40 blocks. Read it for where the counts
stop tapering and jump: a pile-up at or near 1.0 suggests the cut-off belongs just below it.

In [ ]:
max_bar_blocks = 40

total = rate_lf.select(pl.len()).collect().item()

distribution_df = (
    rate_lf.filter(rate >= HISTOGRAM_FROM)
    .with_columns(
        pl.when(rate >= 1)
        .then(1.0)
        .otherwise(HISTOGRAM_FROM + ((rate - HISTOGRAM_FROM) / HISTOGRAM_BIN_WIDTH).floor() * HISTOGRAM_BIN_WIDTH)
        .round(4)
        .alias("band_start")
    )
    .group_by("band_start")
    .agg(pl.len().alias("establishment_months"))
    .sort("band_start")
    .with_columns((pl.col("establishment_months") / total).alias("share"))
    .collect()
)
largest_count = distribution_df["establishment_months"].max()
# Small table: row-wise string repeat is fine here.
distribution_df.with_columns(
    pl.col("establishment_months")
    .map_elements(lambda n: "█" * round(n / largest_count * max_bar_blocks), return_dtype=pl.String)
    .alias("bar")
)

## Candidate maximums

For each value in `CANDIDATE_MAX_RATES`, how many establishment-months are plausible (rate above 0 and at
most the maximum) and how many are above it. Zero-rate establishment-months are implausible at every
maximum, so `plausible` and `above_max` never add up to the total. Pick the lowest maximum where raising
it further would add only a handful of establishment-months.

In [ ]:
pl.concat(
    [
        rate_lf.select(
            pl.lit(max_rate).alias("max_turnover_rate"),
            ((rate > 0) & (rate <= max_rate)).sum().alias("plausible"),
            (rate > max_rate).sum().alias("above_max"),
        )
        for max_rate in CANDIDATE_MAX_RATES
    ]
).with_columns(
    (pl.col("plausible") / total).alias("plausible_share"),
    (pl.col("above_max") / total).alias("above_max_share"),
).collect()